In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parents[0]  # importing functions from other folders
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from _data.data_utils import read_in
# from _fitting.fitting_utils import hist_plot, CI_plot, CI_plot_alt, CI_plot_both, plot_posteriors_side_by_side, plot_spline_Bknots
import pymc as pm
import pymc.math as pmm
import arviz as az
from patsy import dmatrix
import nutpie
import time
from IPython.display import display
from pymc.variational.callbacks import CheckParametersConvergence
import io
import base64
import re
import pytensor.tensor as pt
from pytensor.gradient import disconnected_grad


az.style.use("arviz-darkgrid")


if '___laptop' in os.listdir('../'):
    # laptop folder
    folder = "../../_data/p-dengue/"
elif '___server' in os.listdir('../'):
    # server folder
    folder = "../../../../../data/lucaratzinger_data/p_dengue/"

%matplotlib inline
import seaborn as sns

import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

---

## Models

In [2]:
def fig_to_base64(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=150)
    buf.seek(0)
    img_base64 = base64.b64encode(buf.read()).decode("utf-8")
    plt.close(fig)
    return img_base64

In [3]:
def plot_link(x, idata, var_names=['zi_b0', 'zi_b1'], link='logit'):
    x_mean = np.mean(x)
    x_std_dev = np.std(x)
    
    if np.all([v in idata.posterior for v in var_names]):
        b0_samples = idata.posterior[var_names[0]].values.flatten()
        b1_samples = idata.posterior[var_names[1]].values.flatten()

        x0_samples = -b0_samples / (b1_samples) * x_std_dev + x_mean
        link_samples = np.array([b0 + b1 * ((x - x_mean) / x_std_dev) for b0, b1 in zip(b0_samples, b1_samples)])
    elif all(v in idata.posterior for v in ['zi_c', 'zi_b01', 'zi_b11']):
        c_samples = idata.posterior['zi_c'].values.flatten()
        b01_samples = idata.posterior['zi_b01'].values.flatten()
        b11_samples = idata.posterior['zi_b11'].values.flatten()
        x0_samples = c_samples.copy() * x_std_dev + x_mean
        link_samples = np.array([np.maximum(0, ((x - x_mean) / x_std_dev) - c) * (b01 - b11) + b11 * (((x - x_mean) / x_std_dev) - c) for c, b01, b11 in zip(c_samples, b01_samples, b11_samples)])
    elif all(v in idata.posterior for v in ['zi_c', 'zi_b1', 'zi_b1d']):
        c_samples = idata.posterior['zi_c'].values.flatten()
        b1_samples = idata.posterior['zi_b1'].values.flatten()
        b1d_samples = idata.posterior['zi_b1d'].values.flatten()
        x0_samples = c_samples.copy() * x_std_dev + x_mean
        link_samples = np.array([np.maximum(0, ((x - x_mean) / x_std_dev) - c) * b1d + b1 * (((x - x_mean) / x_std_dev) - c) for c, b1, b1d in zip(c_samples, b1_samples, b1d_samples)])
        
    x0_mean = np.mean(x0_samples)
    x0_lower = np.percentile(x0_samples, 2.5)
    x0_upper = np.percentile(x0_samples, 97.5)

    def logit_to_prob(logit):
        return 1 / (1 + np.exp(-logit))
    def probit_to_prob(probit):
        from scipy.special import erf
        return 0.5 * (1 + erf(probit / np.sqrt(2)))
    if link == 'logit':
        link_samples = logit_to_prob(link_samples)
    elif link == 'probit':
        link_samples = probit_to_prob(link_samples)
    
    link_mean = link_samples.mean(axis=0)
    link_lower5 = np.percentile(link_samples, 25, axis=0)
    link_upper5 = np.percentile(link_samples, 75, axis=0)
    link_lower = np.percentile(link_samples, 2.5, axis=0)
    link_upper = np.percentile(link_samples, 97.5, axis=0)

    # reorder
    id = np.argsort(x)
    x = x[id]
    link_mean = link_mean[id]
    link_lower5 = link_lower5[id]
    link_upper5 = link_upper5[id]
    link_lower = link_lower[id]
    link_upper = link_upper[id]
    link_hdi = np.array([link_lower, link_upper]).T
    plt.figure(figsize=(8, 5))

    plt.axvline(x0_mean, color='red', linestyle='--', label='Mean x0')
    plt.axvline(x0_lower, color='red', linestyle=':', label='95% CI x0')
    plt.axvline(x0_upper, color='red', linestyle=':')

    plt.plot(x, link_mean, label='Mean Link', color='blue')
    plt.fill_between(x, link_hdi[:, 0], link_hdi[:, 1], color='blue', alpha=0.3, label='95% HDI')
    plt.xlabel('x')
    plt.ylabel('Link(psi)')
    plt.title(f'Posterior of Inverse {link.capitalize()} vs x')
    plt.legend()
    plt.grid()
    return plt.gcf()

def plot_link_spline(x, idata, stat_name, B, knots, var_names=['zi_b0', 'zi_b1'], link='logit'):
    id = np.argsort(x)
    x = x[id]
    x_mean = np.mean(x)
    x_std_dev = np.std(x)
    
    if np.all([v in idata.posterior for v in var_names]):
        b0_samples = idata.posterior[var_names[0]].values.flatten()
        b1_samples = idata.posterior[var_names[1]].values.flatten()

        x0_samples = -b0_samples / (b1_samples) * x_std_dev + x_mean
        link_samples = np.array([b0 + b1 * ((x - x_mean) / x_std_dev) for b0, b1 in zip(b0_samples, b1_samples)])
    elif all(v in idata.posterior for v in ['zi_c', 'zi_b01', 'zi_b11']):
        c_samples = idata.posterior['zi_c'].values.flatten()
        b01_samples = idata.posterior['zi_b01'].values.flatten()
        b11_samples = idata.posterior['zi_b11'].values.flatten()
        x0_samples = c_samples.copy() * x_std_dev + x_mean
        link_samples = np.array([np.maximum(0, ((x - x_mean) / x_std_dev) - c) * (b01 - b11) + b11 * (((x - x_mean) / x_std_dev) - c) for c, b01, b11 in zip(c_samples, b01_samples, b11_samples)])
    elif all(v in idata.posterior for v in ['zi_c', 'zi_b1', 'zi_b1d']):
        c_samples = idata.posterior['zi_c'].values.flatten()
        b1_samples = idata.posterior['zi_b1'].values.flatten()
        b1d_samples = idata.posterior['zi_b1d'].values.flatten()
        x0_samples = c_samples.copy() * x_std_dev + x_mean
        link_samples = np.array([np.maximum(0, ((x - x_mean) / x_std_dev) - c) * b1d + b1 * (((x - x_mean) / x_std_dev) - c) for c, b1, b1d in zip(c_samples, b1_samples, b1d_samples)])
        
    x0_mean = np.mean(x0_samples)
    x0_lower = np.percentile(x0_samples, 2.5)
    x0_upper = np.percentile(x0_samples, 97.5)

    def logit_to_prob(logit):
        return 1 / (1 + np.exp(-logit))
    def probit_to_prob(probit):
        from scipy.special import erf
        return 0.5 * (1 + erf(probit / np.sqrt(2)))
    if link == 'logit':
        link_samples = logit_to_prob(link_samples)
    elif link == 'probit':
        link_samples = probit_to_prob(link_samples)

    ###
    knots_local = np.array(knots, copy=True)
    B_local = np.array(B, copy=True, order="F")
    data_local = np.array(x, copy=True)
    B_plot = B_local[id]
    
    # Extract posterior samples
    w_samples = idata.posterior[f'w({stat_name})'].stack(draws=("chain", "draw")).values  # (n_basis, n_draws)
    sigma_w_samples = idata.posterior[f'sigma_w({stat_name})'].stack(draws=("chain", "draw")).values  # (n_draws,)

    f_samples = (B_plot @ w_samples)
    s_samples = np.exp(f_samples)
    ###
    link_samples = link_samples * s_samples.T

    link_mean = link_samples.mean(axis=0)
    link_lower5 = np.percentile(link_samples, 25, axis=0)
    link_upper5 = np.percentile(link_samples, 75, axis=0)
    link_lower = np.percentile(link_samples, 2.5, axis=0)
    link_upper = np.percentile(link_samples, 97.5, axis=0)

    

    link_hdi = np.array([link_lower, link_upper]).T
    plt.figure(figsize=(8, 5))
    
    plt.axvline(knots_local, color='green', linestyle='--', label='Knots')

    plt.axvline(x0_mean, color='red', linestyle='--', label='Mean x0')
    plt.axvline(x0_lower, color='red', linestyle=':', label='95% CI x0')
    plt.axvline(x0_upper, color='red', linestyle=':')

    plt.plot(x, link_mean, label='Mean Link', color='blue')
    plt.fill_between(x, link_hdi[:, 0], link_hdi[:, 1], color='blue', alpha=0.3, label='95% HDI')
    plt.xlabel('x')
    plt.ylabel('Link(psi)')
    plt.title(f'Posterior of Multiplicative Effect (Spline, {link.capitalize()} Link)')
    plt.legend()
    plt.grid()
    return plt.gcf()

def plot_exp_spline(x, idata, stat_name, B, knots):
    id = np.argsort(x)
    x = x[id]
    x_mean = np.mean(x)
    x_std_dev = np.std(x)

    ###
    knots_local = np.array(knots, copy=True)
    B_local = np.array(B, copy=True, order="F")
    data_local = np.array(x, copy=True)
    B_plot = B_local[id]
    
    # Extract posterior samples
    w_samples = idata.posterior[f'w({stat_name})'].stack(draws=("chain", "draw")).values  # (n_basis, n_draws)
    sigma_w_samples = idata.posterior[f'sigma_w({stat_name})'].stack(draws=("chain", "draw")).values  # (n_draws,)

    f_samples = (B_plot @ w_samples)
    s_samples = np.exp(f_samples)
    ###
    link_samples = s_samples.T

    link_mean = link_samples.mean(axis=0)
    link_lower5 = np.percentile(link_samples, 25, axis=0)
    link_upper5 = np.percentile(link_samples, 75, axis=0)
    link_lower = np.percentile(link_samples, 2.5, axis=0)
    link_upper = np.percentile(link_samples, 97.5, axis=0)

    link_hdi = np.array([link_lower, link_upper]).T
    plt.figure(figsize=(8, 5))
    
    for k in knots_local:
        plt.axvline(k, color='green', linestyle='--')

    plt.plot(x, link_mean, label='Mean Link', color='blue')
    plt.fill_between(x, link_hdi[:, 0], link_hdi[:, 1], color='blue', alpha=0.3, label='95% HDI')
    plt.xlabel('x')
    plt.ylabel('Multiplicative Effect (Spline)')
    plt.title(f'Posterior of Multiplicative Effect (Spline)')
    plt.legend()
    plt.grid()
    return plt.gcf()

In [4]:
units = {'t2':'C˚', 'rh':'%RH', 'tp':'mm'}
units_log = {'t2':'C˚', 'rh':'%RH', 'tp':'log(m)'}

def abbrev_stat(stat):
    # remove spaces
    s = stat.replace(" ", "")
    
    # lag extraction: "(k)"
    lag = re.search(r"\((\d+)\)", s)
    lag_str = f"({lag.group(1)})" if lag else ""
    
    # check if _log is present
    has_log = "_log" in s
    
    # weighting
    if "pop_weighted" in s:
        w = "p"
    elif "unweighted" in s:
        w = "u"
    else:
        w = ""
    
    # remove weighting and lag, keep everything else
    base = re.sub(r"_?(pop_weighted|unweighted).*", "", s)
    
    # reattach _log if it was in original
    if has_log and not base.endswith("_log"):
        base += "_log"

    return f"{base}_{w}{lag_str}"

def plot_spline_Bknots(idata, stat_name,
                       var, sigma_var, B, data, knots,
                       figsize=(10,5), show_basis=False, basis_scale=1, invert_log=False, centred_w=True):
    # work on local copies to avoid mutating caller data
    knots_local = np.array(knots, copy=True)
    B_local = np.array(B, copy=True, order="F")
    data_local = np.array(data, copy=True)

    index = np.argsort(data_local)
    data_plot = data_local[index]
    B_plot = B_local[index, :]

    # Extract posterior samples
    w_samples = idata.posterior[var].stack(draws=("chain", "draw")).values  # (n_basis, n_draws)
    sigma_w_samples = idata.posterior[sigma_var].stack(draws=("chain", "draw")).values  # (n_draws,)

    # Compute spline contributions for each draw
    if centred_w:
        f_samples = (B_plot @ w_samples)  # (n_plot, n_draws)
    else:
        f_samples = (B_plot @ w_samples) * sigma_w_samples  # (n_plot, n_draws)

    # Compute mean and credible intervals
    f_mean = f_samples.mean(axis=1)
    f_25 = np.percentile(f_samples, 25, axis=1)
    f_75 = np.percentile(f_samples, 75, axis=1)
    f_025 = np.percentile(f_samples, 2.5, axis=1)
    f_975 = np.percentile(f_samples, 97.5, axis=1)

    # Create figure/axes explicitly
    fig, ax = plt.subplots(figsize=figsize)

    # inverse log trasformation
    if (invert_log) & (stat_name[0:2] == 'tp'):
        plot_knots = (np.exp(knots_local) - 1e-6) * 1000
        data_plot = (np.exp(data_plot) - 1e-6) * 1000
    else:
        plot_knots = knots_local

    ax.vlines(plot_knots, ymin=np.min(f_025), ymax=np.max(f_975), label='knots', lw=0.8, alpha=0.7)
    if show_basis:
        for i in range(B_plot.shape[1]):
            ax.plot(data_plot,
                    np.max(f_975) +
                    (np.max(f_975) - np.min(f_025)) * basis_scale * (
                        (B_plot[:, i] - np.min(B_plot[:, i]))/(np.max(B_plot[:, i]) - np.min(B_plot[:, i])) + 0.05),
                        alpha=0.99, linestyle=':')

    # Main lines and ribbons
    ax.plot(data_plot, f_mean, color='red', label='Mean spline effect')
    ax.fill_between(data_plot, f_025, f_25, color='red', alpha=0.3, label='95% CI')
    ax.fill_between(data_plot, f_25, f_75, color='blue', alpha=0.3, label='50% CI')
    ax.fill_between(data_plot, f_75, f_975, color='red', alpha=0.3)

    abbrev_stat_name = abbrev_stat(stat_name)
    if (invert_log) & (stat_name[0:2] == 'tp'):
        abbrev_stat_name = abbrev_stat_name.replace("_log", "")
        xlab = f'{abbrev_stat_name} ({units[stat_name[0:2]]})'
    else:
        xlab = f'{abbrev_stat_name} ({units_log[stat_name[0:2]]})'

    ax.set_xlabel(xlab)
    ax.set_ylabel('Spline contribution')
    # ax.set_ylim(-0.5, 3.5)
    ax.legend()

    return fig

In [5]:
def elpd_to_row(eval_waic, eval_loo, model_name, data_name):
    return {"model_name": model_name,
            "data_name": data_name,
            # WAIC
            "waic": float(eval_waic.elpd_waic),
            #"p_waic": float(eval_waic.p_waic),
            "waic_se": float(eval_waic.se),
            "waic_warning": int(eval_waic.warning),
            # LOO
            "loo": float(eval_loo.elpd_loo),
            #"p_loo": float(eval_loo.p_loo),
            "loo_se": float(eval_loo.se),
            # diagnostics
            "n_pareto_k_bad": int(np.sum(eval_loo.pareto_k>0.7)),
            "n_pareto_k_very_bad": int(np.sum(eval_loo.pareto_k>1)),
            "pareto_k_mean": float(eval_loo.pareto_k.mean())}

def go(m, model_dict, idata_dict, time_dict, B_dict,
       knot_list_dict, link_dict, stat_names_dict, var_names_dict, n_divergences_dict,
       tune=1000, draws=4000, target_accept=0.8, max_treedepth=10, max_energy_error=1000, compute_idata=True,
       show = {'summary': True, 'trace': True, 'pair': True, 'metrics': True,
               'spline': True, 'exp_spline': True, 'link': True, 'link_spline': True,
               'divergences': True}):
    
    var_names = var_names_dict[m]
    stat_names = stat_names_dict[m]
    with model_dict[m]:
        if compute_idata:
            s0 = time.time()
            idata_dict[m] = pm.sample(
                tune=tune,
                draws=draws,
                chains=4,
                discard_tuned_samples=True,
                store_divergences=True,
                nuts_sampler="nutpie",
                target_accept = target_accept,
                max_treedepth = max_treedepth,
                nuts_sampler_kwargs={"max_energy_error": max_energy_error}
            )
            s1 = time.time()
            pm.compute_log_likelihood(idata_dict[m], progressbar=False)
            s2 = time.time()
            time_dict[m] = (s1 - s0, s2 - s1)
            n_divergences = int(idata_dict[m].sample_stats["diverging"].sum())
            n_divergences_dict[m] = n_divergences
    n_divergences = n_divergences_dict[m]
    # ---------- Summary table ----------
    if show['summary']:
        summary_df = az.summary(idata_dict[m], var_names=var_names)
        summary_html = summary_df.to_html()

    # ---------- Trace plot ----------
    if show['trace']:
        tp_var_names = [v for v in var_names if 'true_scale' not in v]
        fig_trace = az.plot_trace(idata_dict[m], var_names=tp_var_names)
        fig_trace = fig_trace.ravel()[0].figure
        trace_img = fig_to_base64(fig_trace)
    # ---------- Pair plot ----------
    if show['pair']:
        az.rcParams["plot.max_subplots"] = 200
        pp_var_names = [v for v in var_names if 'true_scale' not in v]
        ax = az.plot_pair(
            idata_dict[m],
            var_names=pp_var_names,
            textsize=14,
            divergences=True)

        for i in range(ax.shape[0]):
            ax[i, 0].yaxis.label.set_rotation(0)
            ax[i, 0].yaxis.label.set_ha('right')
        for j in range(ax.shape[1]):
            ax[0, j].xaxis.label.set_rotation(45)
            ax[0, j].xaxis.label.set_ha('right')
        fig_pair = ax.ravel()[0].figure
        pair_img = fig_to_base64(fig_pair)

    #### WAIC and PSIS LOO
    if show['metrics']:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            eval_waic = az.waic(idata_dict[m])
            eval_psis_loo_elpd = az.loo(idata_dict[m])
        wl_df = pd.DataFrame([elpd_to_row(eval_waic, eval_psis_loo_elpd, m, 'd')])
        wl_html = wl_df.to_html()

    # ---------- Spline plot ----------
    if show['spline']:
        spline_imgs = []
        if stat_names is not None:
            for stat_name in stat_names:
                fig_spline = plot_spline_Bknots(
                    idata_dict[m],
                    stat_name,
                    f'w({stat_name})',
                    f'sigma_w({stat_name})',
                    B_dict[m][stat_name],
                    data[stat_name].values,
                    knots=knot_list_dict[m][stat_name],
                    show_basis=True,
                    invert_log=True,
                    centred_w=True
                )
                spline_imgs.append(fig_to_base64(fig_spline))
    if show['exp_spline']:
        exp_spline_imgs = []
        if stat_names is not None:
            for stat_name in stat_names:
                fig_exp_spline = plot_exp_spline(
                    data[stat_name].values,
                    idata_dict[m],
                    stat_name,
                    B_dict[m][stat_name],
                    knot_list_dict[m][stat_name]
                )
                exp_spline_imgs.append(fig_to_base64(fig_exp_spline))
    
    if show['link']:
        if link_dict[m] is not None:
            link = link_dict[m]['link']
            link_stat_name = link_dict[m]['link_stat_name']
            zi_img = fig_to_base64(plot_link(data[link_stat_name].values, idata_dict[m],
                                            var_names=['zi_b0', 'zi_b1'], link=link))
        else:
            zi_img = None

    if show['link_spline']:
        if link_dict[m] is not None:
            if link_dict[m]['link_stat_name'] in stat_names:
                zi_s_img = fig_to_base64(plot_link_spline(data[link_dict[m]['link_stat_name']].values, idata_dict[m],
                                                    link_dict[m]['link_stat_name'], B_dict[m][link_dict[m]['link_stat_name']],
                                                    knot_list_dict[m][link_dict[m]['link_stat_name']], link=link_dict[m]['link']))
        else:
            zi_s_img = None

    #---Divergences plot---
    if show['divergences']:
        if n_divergences > 0:
            posterior = idata_dict[m].posterior.to_dataframe().reset_index()
            stats = idata_dict[m].sample_stats.to_dataframe().reset_index()
            df = posterior.merge(stats, on=["chain","draw"])

            #posterior = az.extract(idata_dict[m], combined=True).to_pandas()
            #stats = idata_dict[m].sample_stats.to_dataframe()
            #df = posterior.join(stats)
            sns.pairplot( df, vars=pp_var_names, hue="diverging",
                        corner=True, diag_kind='kde', plot_kws={"alpha":0.5, 's':1}, diag_kws={"common_norm": False})
            div_img = fig_to_base64(plt.gcf())

    # ---------- Build HTML ----------
    html_content = f"""
    <html>
    <head>
        <title>Model Report: {m}</title>
        <style>
            body {{ font-family: Arial; margin: 40px; }}
            h1 {{ margin-bottom: 10px; }}
            img {{ margin-top: 20px; max-width: 100%; }}
            table {{ border-collapse: collapse; }}
            th, td {{ padding: 6px 8px; }}
        </style>
    </head>
    <body>
        <h1>Model Report: {m}</h1>

        <h2>Timing</h2>
        <p>Posterior Sampling: {time_dict[m][0]:.2f} seconds</p>
        <p>Log Likelihood Compute: {time_dict[m][1]:.2f} seconds</p>

        if show['summary']:
            <h2>Summary</h2>
            {summary_html}
            <p>Divergences: {n_divergences} out of {idata_dict[m].posterior.sizes['draw'] * idata_dict[m].posterior.sizes['chain']} samples ({n_divergences / (idata_dict[m].posterior.sizes['draw'] * idata_dict[m].posterior.sizes['chain']) * 100:.2f}%)</p>
        if show['metrics']:
            <h2>WAIC and PSIS LOO</h2>
            {wl_html}
        if show['trace']:
            <h2>Trace Plot</h2>
            <img src="data:image/png;base64,{trace_img}">
        if show['pair']:
            <h2>Pair Plot</h2>
            <img src="data:image/png;base64,{pair_img}">
        if show['spline']:
            <h2>Spline Plot</h2>
            {"".join(f'<img src="data:image/png;base64,{img}">' for img in spline_imgs)}
        if show['exp_spline']:
            <h2>Expected Spline Plot</h2>
            {"".join(f'<img src="data:image/png;base64,{img}">' for img in exp_spline_imgs)}

        if show['link']:
            <h2>ZI link Plot</h2>
            {f'<img src="data:image/png;base64,{zi_img}">' if zi_img is not None else ""}
        if show['link_spline']:
            <h2>ZI link with Spline Plot</h2>
            {f'<img src="data:image/png;base64,{zi_s_img}">' if zi_s_img is not None else ""}


        if show['divergences']:
            <h2>Divergences Plot</h2>
            <p>Divergences: {n_divergences} out of {idata_dict[m].posterior.sizes['draw'] * idata_dict[m].posterior.sizes['chain']} samples ({n_divergences / (idata_dict[m].posterior.sizes['draw'] * idata_dict[m].posterior.sizes['chain']) * 100:.2f}%)</p>
            {f'<img src="data:image/png;base64,{div_img}">' if n_divergences > 0 else "<p>No divergences detected.</p>"}

    </body>
    </html>
    """

    with open(f"{m}.html", "w") as f:
        f.write(html_content)

    print(f"Saved report to {m}.html")


In [6]:
def go(m, model_dict, idata_dict, time_dict, B_dict,
       knot_list_dict, link_dict, stat_names_dict, var_names_dict, n_divergences_dict,
       tune=1000, draws=4000, target_accept=0.8, max_treedepth=10, max_energy_error=1000, compute_idata=True,
       show = {'summary': True, 'trace': True, 'pair': True, 'metrics': True,
               'spline': True, 'exp_spline': True, 'link': True, 'link_spline': True,
               'divergences': True}):
    
    var_names = var_names_dict[m]
    stat_names = stat_names_dict[m]
    with model_dict[m]:
        if compute_idata:
            s0 = time.time()
            idata_dict[m] = pm.sample(
                tune=tune,
                draws=draws,
                chains=4,
                discard_tuned_samples=True,
                store_divergences=True,
                nuts_sampler="nutpie",
                target_accept = target_accept,
                max_treedepth = max_treedepth,
                nuts_sampler_kwargs={"max_energy_error": max_energy_error}
            )
            s1 = time.time()
            pm.compute_log_likelihood(idata_dict[m], progressbar=False)
            s2 = time.time()
            time_dict[m] = (s1 - s0, s2 - s1)
            n_divergences = int(idata_dict[m].sample_stats["diverging"].sum())
            n_divergences_dict[m] = n_divergences
    n_divergences = n_divergences_dict[m]
    # ---------- Summary table ----------
    if show['summary']:
        summary_df = az.summary(idata_dict[m], var_names=var_names)
        summary_html = summary_df.to_html()

    # ---------- Trace plot ----------
    if show['trace']:
        tp_var_names = [v for v in var_names if 'true_scale' not in v]
        fig_trace = az.plot_trace(idata_dict[m], var_names=tp_var_names)
        fig_trace = fig_trace.ravel()[0].figure
        trace_img = fig_to_base64(fig_trace)
    # ---------- Pair plot ----------
    if show['pair']:
        az.rcParams["plot.max_subplots"] = 200
        pp_var_names = [v for v in var_names if 'true_scale' not in v]
        ax = az.plot_pair(
            idata_dict[m],
            var_names=pp_var_names,
            textsize=14,
            divergences=True)

        for i in range(ax.shape[0]):
            ax[i, 0].yaxis.label.set_rotation(0)
            ax[i, 0].yaxis.label.set_ha('right')
        for j in range(ax.shape[1]):
            ax[0, j].xaxis.label.set_rotation(45)
            ax[0, j].xaxis.label.set_ha('right')
        fig_pair = ax.ravel()[0].figure
        pair_img = fig_to_base64(fig_pair)

    #### WAIC and PSIS LOO
    if show['metrics']:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            eval_waic = az.waic(idata_dict[m])
            eval_psis_loo_elpd = az.loo(idata_dict[m])
        wl_df = pd.DataFrame([elpd_to_row(eval_waic, eval_psis_loo_elpd, m, 'd')])
        wl_html = wl_df.to_html()

    # ---------- Spline plot ----------
    if show['spline']:
        spline_imgs = []
        if stat_names is not None:
            for stat_name in stat_names:
                fig_spline = plot_spline_Bknots(
                    idata_dict[m],
                    stat_name,
                    f'w({stat_name})',
                    f'sigma_w({stat_name})',
                    B_dict[m][stat_name],
                    data[stat_name].values,
                    knots=knot_list_dict[m][stat_name],
                    show_basis=True,
                    invert_log=True,
                    centred_w=True
                )
                spline_imgs.append(fig_to_base64(fig_spline))
    if show['exp_spline']:
        exp_spline_imgs = []
        if stat_names is not None:
            for stat_name in stat_names:
                fig_exp_spline = plot_exp_spline(
                    data[stat_name].values,
                    idata_dict[m],
                    stat_name,
                    B_dict[m][stat_name],
                    knot_list_dict[m][stat_name]
                )
                exp_spline_imgs.append(fig_to_base64(fig_exp_spline))
    
    if show['link']:
        if link_dict[m] is not None:
            link = link_dict[m]['link']
            link_stat_name = link_dict[m]['link_stat_name']
            zi_img = fig_to_base64(plot_link(data[link_stat_name].values, idata_dict[m],
                                            var_names=['zi_b0', 'zi_b1'], link=link))
        else:
            zi_img = None

    if show['link_spline']:
        if (link_dict[m] is not None)&(stat_names is not None):
            if link_dict[m]['link_stat_name'] in stat_names:
                zi_s_img = fig_to_base64(plot_link_spline(data[link_dict[m]['link_stat_name']].values, idata_dict[m],
                                                    link_dict[m]['link_stat_name'], B_dict[m][link_dict[m]['link_stat_name']],
                                                    knot_list_dict[m][link_dict[m]['link_stat_name']], link=link_dict[m]['link']))
        else:
            zi_s_img = None

    #---Divergences plot---
    if show['divergences']:
        if n_divergences > 0:
            posterior = idata_dict[m].posterior.to_dataframe().reset_index()
            stats = idata_dict[m].sample_stats.to_dataframe().reset_index()
            df = posterior.merge(stats, on=["chain","draw"])

            #posterior = az.extract(idata_dict[m], combined=True).to_pandas()
            #stats = idata_dict[m].sample_stats.to_dataframe()
            #df = posterior.join(stats)
            sns.pairplot( df, vars=pp_var_names, hue="diverging",
                        corner=True, diag_kind='kde', plot_kws={"alpha":0.5, 's':1}, diag_kws={"common_norm": False})
            div_img = fig_to_base64(plt.gcf())

    # ---------- Build HTML ----------
    html_parts = []

    # --- Precompute reusable values ---
    total_samples = (
        idata_dict[m].posterior.sizes['draw'] *
        idata_dict[m].posterior.sizes['chain']
    )
    div_pct = n_divergences / total_samples * 100

    # --- Header ---
    html_parts.append(f"""
    <html>
    <head>
        <title>Model Report: {m}</title>
        <style>
            body {{ font-family: Arial; margin: 40px; }}
            h1 {{ margin-bottom: 10px; }}
            img {{ margin-top: 20px; max-width: 100%; }}
            table {{ border-collapse: collapse; }}
            th, td {{ padding: 6px 8px; }}
        </style>
    </head>
    <body>
        <h1>Model Report: {m}</h1>

        <h2>Timing</h2>
        <p>Posterior Sampling: {time_dict[m][0]:.2f} seconds</p>
        <p>Log Likelihood Compute: {time_dict[m][1]:.2f} seconds</p>
    """)

    # --- Summary ---
    if show['summary']:
        html_parts.append(f"""
        <h2>Summary</h2>
        {summary_html}
        <p>Divergences: {n_divergences} out of {total_samples} samples ({div_pct:.2f}%)</p>
        """)

    # --- Metrics ---
    if show['metrics']:
        html_parts.append(f"""
        <h2>WAIC and PSIS LOO</h2>
        {wl_html}
        """)

    # --- Trace ---
    if show['trace']:
        html_parts.append(f"""
        <h2>Trace Plot</h2>
        <img src="data:image/png;base64,{trace_img}">
        """)

    # --- Pair ---
    if show['pair']:
        html_parts.append(f"""
        <h2>Pair Plot</h2>
        <img src="data:image/png;base64,{pair_img}">
        """)

    # --- Spline ---
    if show['spline']:
        imgs = "".join(f'<img src="data:image/png;base64,{img}">' for img in spline_imgs)
        html_parts.append(f"""
        <h2>Spline Plot</h2>
        {imgs}
        """)

    # --- Exponential Spline ---
    if show['exp_spline']:
        imgs = "".join(f'<img src="data:image/png;base64,{img}">' for img in exp_spline_imgs)
        html_parts.append(f"""
        <h2>Exponential Spline Plot</h2>
        {imgs}
        """)

    # --- Link ---
    if show['link']:
        img_html = f'<img src="data:image/png;base64,{zi_img}">' if zi_img is not None else ""
        html_parts.append(f"""
        <h2>ZI link Plot</h2>
        {img_html}
        """)

    # --- Link spline ---
    if show['link_spline']:
        img_html = f'<img src="data:image/png;base64,{zi_s_img}">' if zi_s_img is not None else ""
        html_parts.append(f"""
        <h2>ZI link with Spline Plot</h2>
        {img_html}
        """)

    # --- Divergences ---
    if show['divergences']:
        img_html = (
            f'<img src="data:image/png;base64,{div_img}">'
            if n_divergences > 0
            else "<p>No divergences detected.</p>"
        )
        html_parts.append(f"""
        <h2>Divergences Plot</h2>
        <p>Divergences: {n_divergences} out of {total_samples} samples ({div_pct:.2f}%)</p>
        {img_html}
        """)

    # --- Footer ---
    html_parts.append("""
    </body>
    </html>
    """)

    html_content = "".join(html_parts)

    with open(f"{m}.html", "w") as f:
        f.write(html_content)

    print(f"Saved report to {m}.html")

In [7]:
def difference_matrix(n, order=1):
    """
    Construct a kth-order finite difference matrix of size (n-order, n).

    Parameters
    ----------
    n : int
        Length of the coefficient vector.
    order : int
        Order of the difference.

    Returns
    -------
    D : ndarray
        Difference matrix of shape (n-order, n)
    """
    if order < 1:
        raise ValueError("order must be >= 1")
    if order >= n:
        raise ValueError("order must be < n")

    D = np.eye(n)
    for _ in range(order):
        D = np.diff(D, axis=0)
    return D

In [8]:
def compare_models(models_list, iter):
    # Compare models
    times = {m: [] for m in models_list}
    for m in models_list:
        for i in range(iter):
            with model_dict[m]:
                s0 = time.time()
                idata_dict[m] = pm.sample(tune=1000, draws=4000, chains=4, progressbar=False,
                                            discard_tuned_samples=True, nuts_sampler="nutpie", store_divergences=True)
                s1 = time.time()
                pm.compute_log_likelihood(idata_dict[m], progressbar=False)
                s2 = time.time()
                times[m].append(s1 - s0)
        print(f"Model {m}: Mean time = {np.mean(times[m]):.2f} seconds")
    mean_times = {m: np.mean(times[m]) for m in models_list}
    return times, mean_times 

from multiprocessing import Pool
import numpy as np
import time
import pymc as pm

def _compare_worker(task):
    m, iter = task
    times, mean_times = compare_models([m], iter)
    return m, times[m], mean_times[m]


def compare_models_parallel(models_list, iter, n_workers=4):

    tasks = [(m, iter) for m in models_list]
    with Pool(n_workers) as p:
        results = p.map(_compare_worker, tasks)

    # Reassemble results
    times = {}
    mean_times = {}

    for m, t, mt in results:
        times[m] = t
        mean_times[m] = mt
    return times, mean_times

---

In [9]:
data = read_in(folder, admin=2, max_lag=6, dropna=True, end_year=2017, end_month=12)
# data.to_csv('short_data.csv', index=False)
# data = pd.read_csv('short_data.csv')

In [10]:
model_dict = {}
B_dict = {}
idata_dict = {}
time_dict = {}
knot_list_dict = {}
link_dict = {}
stat_names_dict = {}
var_names_dict = {}
n_divergences_dict = {}

In [11]:
intercept_sigma = 1.0
intercept_mu = -10.0
beta_u_sigma = 1.0
disp_sigma = 1.0

urbanisation_name = 'urbanisation_pop_weighted_std'
knot_type = 'quantile'
degree = 3
num_knots = 5

sigma_w_nu = 1.0
sigma_w_sigma = 1.0

In [12]:
def build_var_names(alpha_parameters, intercept_parameters, link, link_type,
                    b1_parameters, c_parameters, stat_names, penalty_order,
                    penalty_parameters, beta_u_parameters=None):
    var_names = []

    # Always present
    if intercept_parameters is not None:
        var_names.append('intercept')
    if alpha_parameters is not None:
        var_names.append('alpha')

    # Urbanisation
    if beta_u_parameters is not None:
        var_names.append('beta_u')

    # Spline weights
    if stat_names is not None:
        for stat_name in stat_names:
            var_names.append(f'sigma_w({stat_name})')
            var_names.append(f'w({stat_name})')
        if penalty_order is not None:
            for stat_name in stat_names:
                var_names.append(f'lam_({stat_name})')
                var_names.append(f'lam({stat_name})')

    # Zero-inflation / link variables
    if link == 'logit':
        if b1_parameters is not None:
            var_names.append('zi_b1')
        if c_parameters is not None:
            var_names.append('zi_c')
        var_names.append('zi_b0')
        var_names.append('zi_c_true_scale')

    return var_names

In [13]:
def build_model_name(alpha_type, alpha_parameters,
                     intercept_type, intercept_parameters,
                     link, link_stat_name, link_type,
                     b1_type, b1_parameters,
                     c_type, c_parameters,
                     stat_names, num_knots, knot_type, degree,
                     spline_implementation, spline_type, spline_parameters,
                     penalty_order, penalty_type, penalty_parameters, penalty_std,
                     cutoff, beta_u_type, beta_u_parameters,
                     exclude=None):

    exclude = exclude or []

    def fmt_params(params):
        return "(" + ",".join(f"{k}={v}" for k, v in params.items()) + ")"

    parts = []

    # Spline stats
    if stat_names is not None and 'stats' not in exclude:
        stats_str = "+".join(stat_names)
        parts.append(f"stats[{stats_str}]")
        parts.append(f"knots({num_knots},{knot_type},deg={degree},{spline_implementation},spline={spline_type}{',' + fmt_params(spline_parameters) if spline_parameters else ''})")

    # Penalty
    if penalty_order is not None and 'penalty' not in exclude:
        pen_str = f"pen(ord={penalty_order}"
        if penalty_std is not None:
            pen_str += f",std={penalty_std}"
        pen_str += f",type={penalty_type}"
        if penalty_type == 'halfnormal' and penalty_parameters is not None:
            pen_str += "," + ",".join(f"{k}={v}" for k, v in penalty_parameters.items())
        pen_str += ")"
        parts.append(pen_str)

    # Link
    if link is not None and 'link' not in exclude:
        parts.append(f"link({link_stat_name},{link},{link_type},cut={cutoff})")

    # Alpha prior
    if alpha_parameters is not None and 'alpha' not in exclude:
        parts.append(f"alpha({alpha_type},{fmt_params(alpha_parameters)})")

    # Intercept prior
    if intercept_parameters is not None and 'intercept' not in exclude:
        parts.append(f"intercept({intercept_type},{fmt_params(intercept_parameters)})")

    # b1 prior
    if link is not None and b1_parameters is not None and 'b1' not in exclude:
        parts.append(f"b1({b1_type},{fmt_params(b1_parameters)})")

    # c prior
    if link is not None and c_parameters is not None and 'c' not in exclude:
        parts.append(f"c({c_type},{fmt_params(c_parameters)})")

    # Urbanisation prior
    if beta_u_parameters is not None and 'beta_u' not in exclude:
        parts.append(f"betau({beta_u_type},{fmt_params(beta_u_parameters)})")

    return "__".join(parts)

In [14]:
def build_sig_spline_p_model(data,
                             alpha_type, alpha_parameters,
                             intercept_type, intercept_parameters,
                             beta_u_type, beta_u_parameters,
                             link, link_stat_name, link_type,
                             b1_type, b1_parameters,
                             c_type, c_parameters,
                             stat_names, num_knots, knot_type, degree,
                             spline_implementation, spline_type, spline_parameters,
                             penalty_order, penalty_type, penalty_parameters, penalty_std,
                             cutoff,
                             model_dict, B_dict, knot_list_dict, stat_names_dict, var_names_dict, link_dict, 
                             exclude=None):

    m = build_model_name(alpha_type, alpha_parameters,
                        intercept_type, intercept_parameters,
                        link, link_stat_name, link_type,
                        b1_type, b1_parameters,
                        c_type, c_parameters,
                        stat_names, num_knots, knot_type, degree,
                        spline_implementation, spline_type, spline_parameters,
                        penalty_order, penalty_type, penalty_parameters, penalty_std,
                        cutoff, beta_u_type, beta_u_parameters,
                        exclude=exclude)

    model = pm.Model()
    with model:
        # Priors
        if alpha_type == 'exponential':
            alpha = pm.Exponential("alpha", lam=alpha_parameters['lam'])
        elif alpha_type == 'gamma':
            alpha = pm.Gamma("alpha", alpha=alpha_parameters['a'], beta=alpha_parameters['b'])
        if intercept_type == 'normal':
            intercept = pm.Normal("intercept", mu=intercept_parameters['mu'], sigma=intercept_parameters['sigma'])
        if urbanisation_name is not None:
            beta_u = pm.Normal("beta_u", mu=beta_u_parameters['mu'], sigma=beta_u_parameters['sigma'])
        
        # splines
        B = None
        knot_list = None
        if stat_names is not None:
            knot_list = {}
            B = {}
            sigma_w = {}
            w = {}
            f = {}
            for stat_name in stat_names:
                d = data[stat_name].values
                if stat_name == link_stat_name:
                    d = np.clip(d, cutoff, None)
                if knot_type=='equispaced':
                    knot_list[stat_name] = np.linspace(np.min(d), np.max(d), num_knots+2)[1:-1]
                elif knot_type=='quantile':
                    knot_list[stat_name] = np.percentile(np.unique(d), np.linspace(0, 100, num_knots + 2))[1:-1]
                else:
                    print('knot_list must be quantile or equispaced')

                B_full = dmatrix(f"bs(s, knots=knots, degree=degree, include_intercept=True)-1",
                        {"s": d, "knots": knot_list[stat_name], "degree":degree})
                if spline_implementation == 'svd':
                    B_full_centred = B_full - B_full.mean(axis=0)  # centre the spline basis functions
                    U, S, Vt = np.linalg.svd(B_full_centred, full_matrices=False)
                    k = len(S)
                    r = np.sum(S > 1e-10)
                    U_r = U[:, :r]
                    S_r = S[:r]
                    Vt_r = Vt[:r, :]
                    X_r = U_r @ np.diag(S_r)
                    X_r = np.ascontiguousarray(X_r)  # ensure X_r is C-contiguous for PyMC
                    B[stat_name] = X_r

                    # Spline coefficients
                    if spline_type == 'halfnormal':
                        sigma_w[stat_name] = pm.HalfNormal(f"sigma_w({stat_name})", sigma=spline_parameters['sigma_w_sigma'])
                    elif spline_type == 'halfstudentt':
                        sigma_w[stat_name] = pm.HalfStudentT(f"sigma_w({stat_name})", nu=spline_parameters['sigma_w_nu'], sigma=spline_parameters['sigma_w_sigma'])
                    w[stat_name] = pm.Normal(f"w({stat_name})", mu=0, sigma=sigma_w[stat_name], size=B[stat_name].shape[1], dims="splines")

                    if penalty_order is not None:
                        if penalty_type == 'halfnormal':
                            lam_ = pm.HalfNormal(f"lam_({stat_name})", sigma=1.0)
                            h = (np.max(d) - np.min(d)) / (num_knots + 1)
                            lam = pm.Deterministic(f'lam({stat_name})', lam_ * penalty_parameters['sigma'] * h)

                        D = difference_matrix(k, order=penalty_order)
                        DV = D @ Vt_r.T
                        DV = np.ascontiguousarray(DV)
                        DV = pt.as_tensor_variable(DV)
                        # normalise w by its standard deviation to make the penalty scale-invariant
                        if penalty_std:
                            w_std = pt.std(w[stat_name])
                            Dw = pt.dot(DV, w[stat_name] / w_std)
                        else:
                            Dw = pt.dot(DV, w[stat_name])
                        pm.Potential(f"spline_penalty({stat_name})", - lam * pt.dot(Dw, Dw))
                
                    f[stat_name] = pm.math.dot(B[stat_name], w[stat_name])

        # Link
        log_mu = intercept + pm.math.log(data['population'])
        surveillance_name = None
        if surveillance_name is not None:
            log_mu += pm.math.log(pm.math.max(data[surveillance_name], pm.math.log(1e-3)))
        if urbanisation_name is not None:
            log_mu += beta_u*data[urbanisation_name]
        if stat_names is not None:
            for stat_name in stat_names:
                log_mu += f[stat_name]

        # Zero-inflation component
        if link is None:
            y_obs = pm.NegativeBinomial('y_obs', mu=pm.math.exp(log_mu), alpha=alpha, observed=data['cases'])
        else:
            x = data[link_stat_name].values
            x_mean = np.mean(x)
            x_std_dev = np.std(x)
            x_std = (x - x_mean) / x_std_dev
            if link == 'logit':
                if b1_type == 'normal':
                    zi_b1 = pm.Normal("zi_b1", mu=b1_parameters['mu'], sigma=b1_parameters['sigma'])
                if c_type == 'normal':
                    zi_c = pm.Normal("zi_c", mu=c_parameters['mu'], sigma=c_parameters['sigma'])
                zi_b0 = pm.Deterministic("zi_b0", -zi_c * zi_b1)
                zi_x = zi_b0 + zi_b1 * x_std

                #zi_b0_true_scale = pm.Deterministic("zi_b0_true_scale", - zi_b1/x_std_dev*(zi_c*x_std_dev)+x_mean)
                #zi_b1_true_scale = pm.Deterministic("zi_b1_true_scale", zi_b1 / x_std_dev)
                zi_c_true_scale = pm.Deterministic("zi_c_true_scale", zi_c * x_std_dev + x_mean)
            
                # Likelihood
                if link_type == 'multiplicative':
                    y_obs = pm.NegativeBinomial('y_obs', mu=pm.math.invlogit(zi_x) * pm.math.exp(log_mu), alpha=alpha, observed=data['cases'])
                elif link_type == 'additive':
                    y_obs = pm.ZeroInflatedNegativeBinomial('y_obs', psi=pm.math.invlogit(zi_x), mu=pm.math.exp(log_mu), alpha=alpha, observed=data['cases'])

    model_dict[m] = model
    B_dict[m] = B
    knot_list_dict[m] = knot_list
    if link is not None:
        link_dict[m] = {'link': link, 'link_stat_name': link_stat_name}
    else:
        link_dict[m] = None
    stat_names_dict[m] = stat_names
    var_names_dict[m] = build_var_names(alpha_parameters, intercept_parameters, link, link_type,
                                        b1_parameters, c_parameters, stat_names, penalty_order,
                                        penalty_parameters, beta_u_parameters)
    
    return model, m

In [54]:
model, m = build_sig_spline_p_model(data,
                                    alpha_type='exponential', alpha_parameters={'lam': 1.0},
                                    intercept_type='normal', intercept_parameters={'mu': -10.0, 'sigma': 1.0},
                                    beta_u_type='normal', beta_u_parameters={'mu': 0, 'sigma': 1.0},
                                    link='logit', link_stat_name='t2m_mean_pop_weighted(0)', link_type='multiplicative',
                                    b1_type='normal', b1_parameters={'mu': 0, 'sigma': 10.0},
                                    c_type='normal', c_parameters={'mu': 0, 'sigma': 1.0},
                                    stat_names=['t2m_mean_pop_weighted(0)'], num_knots=5, knot_type='equispaced', degree=3, spline_implementation='svd',
                                    penalty_order=2, penalty_type='halfnormal', penalty_parameters={'sigma': 1.0},
                                    cutoff=0.0,
                                    model_dict=model_dict, B_dict=B_dict, knot_list_dict=knot_list_dict,
                                    stat_names_dict=stat_names_dict, var_names_dict=var_names_dict, link_dict=link_dict,
                                    exclude=['alpha', 'intercept', 'beta_u', 'b1'])
m

'stats[t2m_mean_pop_weighted(0)]__knots(5,equispaced,deg=3,svd)__pen(ord=2,halfnormal,sigma=1.0)__link(t2m_mean_pop_weighted(0),logit,multiplicative,cut=0.0)__c(normal,(mu=0,sigma=1.0))'

In [27]:
model, m = build_sig_spline_p_model(data,
                                    alpha_type='gamma', alpha_parameters={'a': 2.0, 'b': 0.1},
                                    intercept_type='normal', intercept_parameters={'mu': -10.0, 'sigma': 1.0},
                                    beta_u_type='normal', beta_u_parameters={'mu': 0, 'sigma': 1.0},
                                    link=None, link_stat_name=None, link_type=None,
                                    b1_type=None, b1_parameters=None,
                                    c_type=None, c_parameters=None,
                                    stat_names=['t2m_mean_pop_weighted(0)'], num_knots=20, knot_type='equispaced', degree=3,
                                    spline_implementation='svd', spline_type='halfstudentt', spline_parameters={'sigma_w_nu': 1.0, 'sigma_w_sigma': 10.0 },
                                    penalty_order=None, penalty_type=None, penalty_parameters=None,
                                    cutoff=None,
                                    model_dict=model_dict, B_dict=B_dict, knot_list_dict=knot_list_dict,
                                    stat_names_dict=stat_names_dict, var_names_dict=var_names_dict, link_dict=link_dict,
                                    exclude=['intercept', 'beta_u'])
print('model ', m)
go(m, model_dict, idata_dict, time_dict, B_dict,
   knot_list_dict, link_dict, stat_names_dict, var_names_dict, n_divergences_dict,
   tune=1000, draws=2000, target_accept=0.8, max_treedepth=10, compute_idata=True)

model  stats[t2m_mean_pop_weighted(0)]__knots(20,equispaced,deg=3,svd,spline=halfstudentt,(sigma_w_nu=1.0,sigma_w_sigma=10.0))__alpha(gamma,(a=2.0,b=0.1))


Progress,Draws,Divergences,Step Size,Gradients/Draw
,3000,0,0.49,7
,3000,0,0.49,47
,3000,0,0.49,7
,3000,0,0.40,15


/mnt/b1/lucaratzinger/miniconda3/envs/p-dengue-py311-2/lib/python3.11/site-packages/arviz/plots/backends/matplotlib/pairplot.py:223: UserWarning: rcParams['plot.max_subplots'] (200) is smaller than the number of resulting pair plots with these variables, generating only a 19x19 grid
  warnings.warn(


Saved report to stats[t2m_mean_pop_weighted(0)]__knots(20,equispaced,deg=3,svd,spline=halfstudentt,(sigma_w_nu=1.0,sigma_w_sigma=10.0))__alpha(gamma,(a=2.0,b=0.1)).html


In [16]:
model, m = build_sig_spline_p_model(data,
                                    alpha_type='gamma', alpha_parameters={'a': 2.0, 'b': 0.1},
                                    intercept_type='normal', intercept_parameters={'mu': -10.0, 'sigma': 1.0},
                                    beta_u_type='normal', beta_u_parameters={'mu': 0, 'sigma': 1.0},
                                    link=None, link_stat_name=None, link_type=None,
                                    b1_type=None, b1_parameters=None,
                                    c_type=None, c_parameters=None,
                                    stat_names=['t2m_mean_pop_weighted(0)'], num_knots=20, knot_type='equispaced', degree=3,
                                    spline_implementation='svd', spline_type='halfstudentt', spline_parameters={'sigma_w_nu': 1.0, 'sigma_w_sigma': 10.0 },
                                    penalty_order=2, penalty_type='halfnormal', penalty_parameters={'sigma': 30.0}, penalty_std=True,
                                    cutoff=None,
                                    model_dict=model_dict, B_dict=B_dict, knot_list_dict=knot_list_dict,
                                    stat_names_dict=stat_names_dict, var_names_dict=var_names_dict, link_dict=link_dict,
                                    exclude=['intercept', 'beta_u'])
print('model ', m)
go(m, model_dict, idata_dict, time_dict, B_dict,
   knot_list_dict, link_dict, stat_names_dict, var_names_dict, n_divergences_dict,
   tune=1000, draws=2000, target_accept=0.8, max_treedepth=10, compute_idata=True)

model  stats[t2m_mean_pop_weighted(0)]__knots(20,equispaced,deg=3,svd,spline=halfstudentt,(sigma_w_nu=1.0,sigma_w_sigma=10.0))__pen(ord=2,std=True,type=halfnormal,sigma=30.0)__alpha(gamma,(a=2.0,b=0.1))


Progress,Draws,Divergences,Step Size,Gradients/Draw
,3000,0,0.44,15
,3000,0,0.48,7
,3000,0,0.42,7
,3000,0,0.46,7


/mnt/b1/lucaratzinger/miniconda3/envs/p-dengue-py311-2/lib/python3.11/site-packages/arviz/plots/backends/matplotlib/pairplot.py:223: UserWarning: rcParams['plot.max_subplots'] (200) is smaller than the number of resulting pair plots with these variables, generating only a 19x19 grid
  warnings.warn(


Saved report to stats[t2m_mean_pop_weighted(0)]__knots(20,equispaced,deg=3,svd,spline=halfstudentt,(sigma_w_nu=1.0,sigma_w_sigma=10.0))__pen(ord=2,std=True,type=halfnormal,sigma=30.0)__alpha(gamma,(a=2.0,b=0.1)).html
